# DynaICL Stage 0: remote validation pipeline (Google Colab)

Runs the long, CPU-only validation that is too big for the local machine:

1. the full pytest suite (baseline),
2. the **full mutation-detection matrix**: every validation check, and the whole pytest suite, under every injected bug,
3. optionally, the main separability experiment again (cross-machine check).

**Before you start**
* `Runtime -> Change runtime type -> CPU`. **No GPU is needed**; this is pure numpy/scipy.
* Expected time on the free tier (2 vCPU): about **1-1.5 h**. This is an estimate from local timings, not measured on Colab.
  High-CPU runtimes are faster (the pytest phase parallelises over cores).
* Keep this tab open while it runs. When it finishes, the results archive is saved to your Google Drive and also downloaded.

**You need:** `dynaicl_stage0_bundle.zip` (SHA-256 `5ace85ebf8bc81b6...`, 83 files). Then `Runtime -> Run all`.

## 1. Runtime check

In [ ]:
!python3 --version
!nproc
!free -h | head -2

## 2. Get the code bundle
Leave `BUNDLE_ON_DRIVE = ""` to upload the zip from your computer when prompted. Or put the zip on Google Drive
and set the path, e.g. `"/content/drive/MyDrive/dynaicl_stage0_bundle.zip"`.

In [ ]:
import hashlib, os, zipfile
BUNDLE_ON_DRIVE = ""          # optional Drive path to dynaicl_stage0_bundle.zip
EXPECTED_SHA256 = "5ace85ebf8bc81b617279312994b69eb130ffb814fb3ca37c8d9d8c28503e117"

if BUNDLE_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    src = BUNDLE_ON_DRIVE
else:
    from google.colab import files
    uploaded = files.upload()          # pick dynaicl_stage0_bundle.zip
    src = next(iter(uploaded))

sha = hashlib.sha256(open(src, "rb").read()).hexdigest()
print("bundle sha256:", sha, "(matches the bundle this notebook was built with)" if sha == EXPECTED_SHA256
      else "WARNING: different bundle than this notebook was built with; continuing")
os.makedirs("/content/DynaICL", exist_ok=True)
zipfile.ZipFile(src).extractall("/content/DynaICL")
os.chdir("/content/DynaICL")

# integrity check of every extracted file against the bundle manifest
bad = []
for line in open("MANIFEST.sha256"):
    h, rel = line.rstrip("\n").split("  ", 1)
    if hashlib.sha256(open(rel, "rb").read()).hexdigest() != h:
        bad.append(rel)
print("manifest check:", "OK" if not bad else f"MISMATCH in {bad}")

## 3. Install the pinned stack
These are the exact versions the suite was validated with locally (numpy 2.2.2, scipy 1.15.1, pyyaml 6.0.2,
pytest 9.0.2). Pip may warn about conflicts with other preinstalled Colab packages. That is harmless here,
because everything runs in fresh subprocesses that use only these packages.

In [ ]:
!pip install -q -r /content/DynaICL/requirements.txt
!python3 -c "import numpy, scipy, yaml, pytest; print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| pyyaml', yaml.__version__, '| pytest', pytest.__version__)"

## 4. Fast smoke test (about 1-2 min): fail early if the environment is broken

In [ ]:
%cd /content/DynaICL
!python3 -m pytest -q -p no:cacheprovider tests/test_c_identities.py tests/test_h_numerics.py tests/test_j_leakage.py tests/test_metrics.py

## 5. The pipeline (the long step)
`RUN_EXPERIMENT=1` also reruns the main separability experiment (about 30+ min extra on 2 vCPU). It is optional.

In [ ]:
%cd /content/DynaICL
!JOBS=$(nproc) RUN_EXPERIMENT=0 bash scripts/remote_validation_pipeline.sh 2>&1 | tee /content/pipeline_log.txt

## 6. Results summary

In [ ]:
%cd /content/DynaICL
!cat remote_results_*/STATUS.txt
!echo; cat remote_results_*/mutation_report_*/matrix.md
!echo; tail -n 3 remote_results_*/pytest_baseline.txt

## 7. Save the results archive (Drive + download)
Send back the `remote_results_<UTC>.tar.gz`. If the download is blocked, it is also in `MyDrive/dynaicl_results/`.

In [ ]:
import glob, os, shutil
%cd /content/DynaICL
archive = sorted(glob.glob("remote_results_*.tar.gz"))[-1]
print("archive:", archive, os.path.getsize(archive), "bytes")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/dynaicl_results", exist_ok=True)
    shutil.copy(archive, "/content/drive/MyDrive/dynaicl_results/")
    shutil.copy("/content/pipeline_log.txt", "/content/drive/MyDrive/dynaicl_results/" + archive.replace(".tar.gz", "_log.txt"))
    print("copied to MyDrive/dynaicl_results/")
except Exception as exc:
    print("Drive copy skipped:", exc)
from google.colab import files
files.download(archive)